# Notebook 01 — Process Real Data
Convert raw Cowrie logs + CIC/UNSW-NB15 into structured session DataFrames.
**Target:** 180,000 real Cowrie + 60,000 transfer = 240,000 real sessions.

In [1]:
import sys, json, logging, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger()

# ── Ensure project root is on path ──
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
DATA_RAW  = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
DATA_PROC.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

Project root: C:\Users\mahit\OneDrive\Desktop\adaptive-honeypot-ml-CAPSTONE\honeypot_dataset


## 1.1  Parse Cowrie JSON Logs

In [2]:
import re, math
from collections import defaultdict
from datetime import datetime, timezone
import pandas as pd
import numpy as np

COWRIE_LOG_DIR = DATA_RAW / "cowrie_logs"

# Map Cowrie event types → micro-state labels
EVENT_MAP = {
    "cowrie.session.connect":       "RECON_IP_SCAN",
    "cowrie.login.failed":          "ACCESS_BRUTE_SSH",
    "cowrie.login.success":         "ACCESS_BRUTE_SSH",
    "cowrie.session.file_download": "EXEC_WGET_EXEC",
    "cowrie.session.file_upload":   "EXFIL_SCP_DATA",
    "cowrie.direct-tcpip.request":  "LATERAL_SSH_SPREAD",
}

COMMAND_PATTERNS = [
    (re.compile(r"\b(uname|hostname|id|whoami|cat /proc)\b"),     "DISC_ENV_PROBE"),
    (re.compile(r"\b(netstat|ss -|ip addr|ifconfig)\b"),           "DISC_NETSTAT_SCAN"),
    (re.compile(r"\bps\s"),                                         "DISC_PROC_ENUM"),
    (re.compile(r"find.*-perm.*4000|-perm.*-u=s"),                   "DISC_SUID_HUNT"),
    (re.compile(r"\b(wget|curl)\b.*http"),                          "EXEC_WGET_EXEC"),
    (re.compile(r"\bpython[23]?\b"),                                "EXEC_PYTHON_SCRIPT"),
    (re.compile(r"\bperl\b"),                                       "EXEC_PERL_SCRIPT"),
    (re.compile(r"bash.*-i.*>&|/dev/tcp|nc.*-e|ncat"),               "EXEC_CURL_BASH"),
    (re.compile(r"\bsudo\b"),                                       "PRIVESC_SUDO_ABUSE"),
    (re.compile(r"\bcrontab\b"),                                    "PERSIST_CRONTAB"),
    (re.compile(r"authorized_keys"),                                  "PERSIST_SSH_KEY_ADD"),
    (re.compile(r"history\s*-c|>\s*/dev/null|rm.*\.bash_history"),"EVASION_HIST_ERASE"),
    (re.compile(r"rm.*(/var/log|/var/run)"),                         "EVASION_LOG_WIPE"),
    (re.compile(r"\bscp\b"),                                        "EXFIL_SCP_DATA"),
    (re.compile(r"\btar\b.*-czf"),                                  "EXFIL_STAGING_TAR"),
]

def label_command(cmd: str) -> str:
    cmd_l = cmd.lower()
    for pattern, label in COMMAND_PATTERNS:
        if pattern.search(cmd_l):
            return label
    return "EXEC_SHELL_OPEN"

def parse_cowrie_file(fpath: Path) -> list:
    sessions = defaultdict(list)
    with open(fpath, encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                ev = json.loads(line)
                sid = ev.get("session","")
                if sid: sessions[sid].append(ev)
            except: continue
    return list(sessions.values())

def build_record(events: list) -> dict | None:
    if not events: return None
    events.sort(key=lambda e: e.get("timestamp",""))
    first, last = events[0], events[-1]

    def to_ts(s):
        try: return datetime.fromisoformat(s.replace("Z","+00:00")).timestamp()
        except: return 0.0

    t_start = to_ts(first.get("timestamp",""))
    t_end   = to_ts(last.get("timestamp",""))
    ts_list = [to_ts(e.get("timestamp","")) for e in events if e.get("timestamp")]

    cmds, logins, downloads, seq = [], 0, [], []
    t_first_auth = t_first_cmd = None

    for ev in events:
        etype = ev.get("eventid","")
        if etype in ("cowrie.login.failed","cowrie.login.success"):
            logins += 1
            seq.append("ACCESS_BRUTE_SSH")
            if t_first_auth is None: t_first_auth = to_ts(ev.get("timestamp",""))
        elif etype == "cowrie.command.input":
            cmd = ev.get("input","")
            cmds.append(cmd)
            seq.append(label_command(cmd))
            if t_first_cmd is None: t_first_cmd = to_ts(ev.get("timestamp",""))
        elif etype == "cowrie.session.file_download":
            downloads.append(ev.get("url",""))
            seq.append("EXEC_WGET_EXEC")
        elif etype == "cowrie.session.file_upload":
            seq.append("EXFIL_SCP_DATA")

    if not seq: seq = ["RECON_IP_SCAN"]
    meaningful = [s for s in seq if not s.startswith("RECON")]
    label = meaningful[-1] if meaningful else seq[-1]

    return {
        "session_id":           first.get("session",""),
        "src_ip":               first.get("src_ip",""),
        "src_port":             first.get("src_port",0),
        "dst_port":             22,
        "protocol":             "ssh",
        "t_start":              t_start,
        "t_end":                t_end,
        "t_first_auth":         t_first_auth or t_start,
        "t_first_cmd":          t_first_cmd  or t_start,
        "event_timestamps":     ts_list,
        "session_duration_s":   max(0.0, t_end - t_start),
        "n_events":             len(events),
        "n_commands":           len(cmds),
        "login_attempts":       logins,
        "command_text":         " ; ".join(cmds) if cmds else "[no commands]",
        "n_downloads":          len(downloads),
        "bytes_in":             sum(e.get("size",0) for e in events if "size" in e),
        "bytes_out":            0,
        "micro_state":          label,
        "micro_state_sequence": ",".join(seq),
        "source":               "cowrie_real",
        "associated_cve":       "",
    }

In [3]:
# ── Parse all Cowrie logs ──
all_records = []
log_files   = sorted(COWRIE_LOG_DIR.glob("cowrie.json*"))
print(f"Found {len(log_files)} Cowrie log files")

if not log_files:
    print("⚠  No Cowrie logs found in", COWRIE_LOG_DIR)
    print("   Place your cowrie.json.YYYY-MM-DD files there and re-run.")
else:
    for fpath in log_files:
        sessions = parse_cowrie_file(fpath)
        recs = [build_record(s) for s in sessions]
        recs = [r for r in recs if r is not None]
        all_records.extend(recs)
        print(f"  {fpath.name}: {len(recs)} sessions")

    df_cowrie = pd.DataFrame(all_records).drop_duplicates("session_id")
    print(f"\nTotal Cowrie sessions: {len(df_cowrie):,}")
    print("\nMicro-state distribution:")
    print(df_cowrie["micro_state"].value_counts().head(20))
    df_cowrie.to_parquet(DATA_PROC / "real_cowrie_raw.parquet", index=False)
    print(f"\nSaved → {DATA_PROC}/real_cowrie_raw.parquet")

Found 0 Cowrie log files
⚠  No Cowrie logs found in C:\Users\mahit\OneDrive\Desktop\adaptive-honeypot-ml-CAPSTONE\honeypot_dataset\data\raw\cowrie_logs
   Place your cowrie.json.YYYY-MM-DD files there and re-run.


## 1.2  Load CIC-IDS2017 Transfer Dataset

In [4]:
# ── CIC-IDS2017 label → micro-state mapping ──
CIC_LABEL_MAP = {
    "BENIGN":              None,             # skip benign traffic
    "DoS Hulk":            "DISC_ENV_PROBE",
    "PortScan":            "RECON_IP_SCAN",
    "DDoS":                "DISC_NETSTAT_SCAN",
    "DoS GoldenEye":       "DISC_ENV_PROBE",
    "FTP-Patator":         "ACCESS_BRUTE_SSH",
    "SSH-Patator":         "ACCESS_BRUTE_SSH",
    "DoS slowloris":       "DISC_ENV_PROBE",
    "DoS Slowhttptest":    "DISC_ENV_PROBE",
    "Bot":                 "PERSIST_CRONTAB",
    "Web Attack Brute Force": "ACCESS_BRUTE_HTTP",
    "Web Attack XSS":      "EXEC_CURL_BASH",
    "Infiltration":        "LATERAL_SSH_SPREAD",
    "Web Attack Sql Injection": "EXEC_SHELL_OPEN",
    "Heartbleed":          "ACCESS_KEX_EXPLOIT",
}

CIC_DIR = DATA_RAW / "cic_ids2017"
cic_files = list(CIC_DIR.glob("*.csv")) + list(CIC_DIR.glob("*.parquet"))

if not cic_files:
    print("⚠  CIC-IDS2017 files not found in", CIC_DIR)
    print("   Download from: https://www.unb.ca/cic/datasets/ids-2017.html")
    df_cic = pd.DataFrame()
else:
    frames = []
    for f in cic_files:
        chunk = pd.read_csv(f, low_memory=False) if f.suffix==".csv" else pd.read_parquet(f)
        chunk.columns = [c.strip() for c in chunk.columns]
        frames.append(chunk)
    df_cic_raw = pd.concat(frames, ignore_index=True)
    label_col  = next((c for c in df_cic_raw.columns if "label" in c.lower()), None)
    if label_col:
        df_cic_raw["micro_state"] = df_cic_raw[label_col].map(CIC_LABEL_MAP)
        df_cic = df_cic_raw[df_cic_raw["micro_state"].notna()].copy()
        df_cic["source"] = "cic_ids2017"
        print(f"CIC sessions after mapping: {len(df_cic):,}")
        print(df_cic["micro_state"].value_counts())
    else:
        print("Could not find label column in CIC dataset")
        df_cic = pd.DataFrame()

CIC sessions after mapping: 555,466
micro_state
DISC_ENV_PROBE        252661
RECON_IP_SCAN         158930
DISC_NETSTAT_SCAN     128027
ACCESS_BRUTE_SSH       13835
PERSIST_CRONTAB         1966
LATERAL_SSH_SPREAD        36
ACCESS_KEX_EXPLOIT        11
Name: count, dtype: int64


In [5]:
# ── 1.2b  Load UNSW-NB15 Transfer Dataset ──
UNSW_LABEL_MAP = {
    "Normal":          None,                   # skip benign traffic
    "Fuzzers":         "DISC_ENV_PROBE",
    "Analysis":        "RECON_VULN_SCAN",
    "Backdoors":       "PERSIST_BACKDOOR_ADD",
    "DoS":             "DISC_NETSTAT_SCAN",
    "Exploits":        "ACCESS_KEX_EXPLOIT",
    "Generic":         "EXEC_SHELL_OPEN",
    "Reconnaissance":  "RECON_IP_SCAN",
    "Shellcode":       "EXEC_MEMFD_EXEC",
    "Worms":           "LATERAL_SSH_SPREAD",
}

UNSW_DIR = DATA_RAW / "unsw_nb15"
unsw_train = UNSW_DIR / "UNSW_NB15_training-set.csv"
unsw_test  = UNSW_DIR / "UNSW_NB15_testing-set.csv"

if unsw_train.exists() and unsw_test.exists():
    df_unsw_raw = pd.concat([
        pd.read_csv(unsw_train),
        pd.read_csv(unsw_test),
    ], ignore_index=True)
    df_unsw_raw.columns = [c.strip() for c in df_unsw_raw.columns]

    label_col = next((c for c in df_unsw_raw.columns
                      if c.lower() in ("attack_cat","attack_category")), None)
    if label_col:
        df_unsw_raw["micro_state"] = df_unsw_raw[label_col].str.strip().map(UNSW_LABEL_MAP)
        df_unsw = df_unsw_raw[df_unsw_raw["micro_state"].notna()].copy()
        df_unsw["source"] = "unsw_nb15"
        print(f"UNSW-NB15 sessions after mapping: {len(df_unsw):,}")
        print(df_unsw["micro_state"].value_counts())
    else:
        print("Could not find attack_cat column in UNSW dataset")
        df_unsw = pd.DataFrame()
else:
    print(f"⚠  UNSW files not found in {UNSW_DIR}")
    df_unsw = pd.DataFrame()

UNSW-NB15 sessions after mapping: 162,344
micro_state
EXEC_SHELL_OPEN       58871
ACCESS_KEX_EXPLOIT    44525
DISC_ENV_PROBE        24246
DISC_NETSTAT_SCAN     16353
RECON_IP_SCAN         13987
RECON_VULN_SCAN        2677
EXEC_MEMFD_EXEC        1511
LATERAL_SSH_SPREAD      174
Name: count, dtype: int64


## 1.3  Combine Real Data

In [6]:
real_frames = []

if "df_cowrie" in dir() and len(df_cowrie) > 0:
    real_frames.append(df_cowrie)

if "df_cic" in dir() and len(df_cic) > 0:
    df_cic = df_cic.reset_index(drop=True)
    n = len(df_cic)

    def describe_cic_flow(row):
        proto = row.get("Protocol", "TCP")
        duration = row.get("Flow Duration", 0)
        fwd_pkts = row.get("Total Fwd Packets", 0)
        bwd_pkts = row.get("Total Backward Packets", 0)
        dst_port = row.get("Destination Port", 0)
        label = row.get("micro_state", "unknown")
        return (f"{proto} flow to port {dst_port}, duration {duration}us, "
                f"{fwd_pkts} forward packets, {bwd_pkts} backward packets, "
                f"classified as {label}")

    df_cic["command_text"] = df_cic.apply(describe_cic_flow, axis=1)

    cic_aligned = pd.DataFrame({
        "session_id":           [f"cic_{i}" for i in range(n)],
        "micro_state":          df_cic["micro_state"].to_numpy(),
        "micro_state_sequence": df_cic["micro_state"].to_numpy(),
        "source":               "cic_transfer",
        "session_duration_s":   df_cic.get("Flow Duration", pd.Series([1.0]*n)).to_numpy() / 1e6,
        "bytes_in":             df_cic.get("Total Fwd Packets", pd.Series([0]*n)).to_numpy(),
        "bytes_out":            df_cic.get("Total Backward Packets", pd.Series([0]*n)).to_numpy(),
        "n_events": 5, "n_commands": 0, "login_attempts": 0,
        "command_text": df_cic["command_text"].to_numpy(),
        "t_start": 1_700_000_000.0, "t_end": 1_700_000_001.0,
        "dst_port": 80, "protocol": "tcp", "associated_cve": "",
    })
    real_frames.append(cic_aligned)

if "df_unsw" in dir() and len(df_unsw) > 0:
    df_unsw = df_unsw.reset_index(drop=True)
    n = len(df_unsw)

    def describe_unsw_flow(row):
        proto = row.get("proto", "tcp")
        duration = row.get("dur", 0)
        sbytes = row.get("sbytes", 0)
        dbytes = row.get("dbytes", 0)
        dsport = row.get("dsport", 0)
        label = row.get("micro_state", "unknown")
        return (f"{proto} flow to port {dsport}, duration {duration}s, "
                f"{sbytes} source bytes, {dbytes} dest bytes, "
                f"classified as {label}")

    df_unsw["command_text"] = df_unsw.apply(describe_unsw_flow, axis=1)

    # UNSW-NB15 train/test set has NO dsport column (that's only in the raw
    # 4-part files). Default to port 22 since most UNSW attack records
    # represent generic network traffic, not service-specific sessions.
    if "dsport" in df_unsw.columns:
        dst_port_col = pd.to_numeric(df_unsw["dsport"], errors="coerce").fillna(22).astype(int).to_numpy()
    else:
        dst_port_col = np.full(n, 22, dtype=int)

    proto_col = (df_unsw["proto"].to_numpy() if "proto" in df_unsw.columns
                else np.full(n, "tcp", dtype=object))

    unsw_aligned = pd.DataFrame({
        "session_id":           [f"unsw_{i}" for i in range(n)],
        "micro_state":          df_unsw["micro_state"].to_numpy(),
        "micro_state_sequence": df_unsw["micro_state"].to_numpy(),
        "source":               "unsw_transfer",
        "session_duration_s":   df_unsw.get("dur",    pd.Series([1.0]*n)).to_numpy(),
        "bytes_in":             df_unsw.get("sbytes",  pd.Series([0]*n)).to_numpy(),
        "bytes_out":            df_unsw.get("dbytes",  pd.Series([0]*n)).to_numpy(),
        "n_events": 5, "n_commands": 0, "login_attempts": 0,
        "command_text": df_unsw["command_text"].to_numpy(),
        "t_start": 1_700_000_000.0, "t_end": 1_700_000_001.0,
        "dst_port": dst_port_col,
        "protocol": proto_col,
        "associated_cve": "",
    })
    real_frames.append(unsw_aligned)

if not real_frames:
    print("⚠  No real data found — generating minimal placeholder")
    from src.generators.kill_chain_simulator import generate_balanced_sessions
    df_real = generate_balanced_sessions(n_total=10_000, seed=42)
    df_real["source"] = "simulated_placeholder"
else:
    df_real = pd.concat(real_frames, ignore_index=True)

print(f"\nTotal real sessions: {len(df_real):,}")
print("Sources:", df_real["source"].value_counts().to_dict())
print("Micro-states:", df_real["micro_state"].nunique(), "unique classes")
print("\ncommand_text.value_counts().head(10):")
print(df_real["command_text"].value_counts().head(10))
df_real.to_parquet(DATA_PROC / "real_sessions_combined.parquet", index=False)
print(f"\nSaved → {DATA_PROC}/real_sessions_combined.parquet")


Total real sessions: 717,810
Sources: {'cic_transfer': 555466, 'unsw_transfer': 162344}
Micro-states: 10 unique classes

command_text.value_counts().head(10):


command_text
TCP flow to port 80, duration 3us, 2 forward packets, 0 backward packets, classified as DISC_ENV_PROBE    16220
udp flow to port 0, duration 9e-06s, 114 source bytes, 0 dest bytes, classified as EXEC_SHELL_OPEN        15264
TCP flow to port 80, duration 4us, 2 forward packets, 0 backward packets, classified as DISC_ENV_PROBE    10955
TCP flow to port 80, duration 1us, 2 forward packets, 0 backward packets, classified as DISC_ENV_PROBE     9226
udp flow to port 0, duration 8e-06s, 114 source bytes, 0 dest bytes, classified as EXEC_SHELL_OPEN         8970
udp flow to port 0, duration 3e-06s, 114 source bytes, 0 dest bytes, classified as EXEC_SHELL_OPEN         8776
udp flow to port 0, duration 5e-06s, 114 source bytes, 0 dest bytes, classified as EXEC_SHELL_OPEN         5050
TCP flow to port 80, duration 2us, 2 forward packets, 0 backward packets, classified as DISC_ENV_PROBE     4040
udp flow to port 0, duration 4e-06s, 114 source bytes, 0 dest bytes, classified as EXEC_SHE


Saved → C:\Users\mahit\OneDrive\Desktop\adaptive-honeypot-ml-CAPSTONE\honeypot_dataset\data\processed/real_sessions_combined.parquet
